In [1]:
import torch
import torch.nn.functional as F

def scaled_dot_prod_attention(q, k, v, mask=None):
    """Implement Scaled Dot Product Attention (Vaswani et al., "Attention Is All You Need")."""
    d_k = q.size(-1)
    scores = torch.matmul(q, k.transpose(-2, -1)) / (d_k ** 0.5)
    if mask is not None:
        scores = scores.masked_fill(mask == 0, float('-inf'))

    attention_weights = F.softmax(scores, dim=-1)
    attention_weights = F.dropout(attention_weights, 0.1)
    output = torch.matmul(attention_weights, v)
    return output, attention_weights


In [ ]:
def subsequential_mask(size):
    "Mask out subsequnt positions"
    attn_shape = (1,size ,size)
    subsequent_mask = torch.triu(torch.ones(attn_shape) , diagonal = 1).type(torch.uint8)
    return subsequent_mask == 0


In [ ]:
# Run attention on dummy Q, K, V and plot attention as colored squares (heatmap)
import matplotlib.pyplot as plt

# Example: seq_len=6, d_k=4, batch=1 — all tensors created with PyTorch
seq_len, d_k = 6, 4
q = torch.randn(1, seq_len, d_k)
k = torch.randn(1, seq_len, d_k)
v = torch.randn(1, seq_len, d_k)

# Causal mask so each position only attends to past
mask = subsequential_mask(seq_len)
output, attn_weights = scaled_dot_prod_attention(q, k, v, mask=mask)

# Keep as PyTorch tensor until plot; (1, seq_len, seq_len) -> (seq_len, seq_len)
attn = attn_weights[0].detach()
# Convert to numpy only for matplotlib imshow
fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(attn.numpy(), cmap='viridis', aspect='equal', vmin=0, vmax=1)
ax.set_xticks(range(seq_len))
ax.set_yticks(range(seq_len))
ax.set_xlabel('Key position')
ax.set_ylabel('Query position')
ax.set_title('Attention weights (colored squares)')
plt.colorbar(im, ax=ax, label='Weight')
plt.tight_layout()
plt.show()